In [ ]:
!pip install pandas openpyxl
!pip install requests
!pip install beautifulsoup4

In [ ]:
import os
from datetime import datetime

import pandas as pd

import requests
from bs4 import BeautifulSoup

URL_BASE = "https://dom.pmt.pi.gov.br"
URL_INICIO = "https://dom.pmt.pi.gov.br/lista_diario.php"

CAMINHO_RAIZ = os.getcwd()
CAMINHO_DOWNLOAD = os.path.join(CAMINHO_RAIZ, "downloads")
os.makedirs(CAMINHO_DOWNLOAD, exist_ok=True)

DATA_LOG = datetime.now().strftime("%Y_%m_%d")

In [ ]:
contador = 1
dados_diarios = []

sessao = requests.Session()

resposta = sessao.get(URL_INICIO)

while True:
    print(f"Executando para a página {contador}")
    soup = BeautifulSoup(resposta.content)

    tabela = soup.find(class_="table table-bordered table-striped").find("tbody")
    linhas = tabela.find_all("tr")

    for linha in linhas:
        colunas = linha.find_all("td")

        numero = colunas[0].get_text()
        data = colunas[1].get_text()
        arquivo = colunas[2].find("a")
        url_arquivo = arquivo["href"]
        titulo_arquivo = arquivo.get_text().strip()

        caminho_arquivo = url_arquivo.split("/")[-1]
        caminho_arquivo = os.path.join(CAMINHO_DOWNLOAD, caminho_arquivo)

        resposta = requests.get(url_arquivo)
        with open(caminho_arquivo, "wb") as arquivo_baixado:
            arquivo_baixado.write(resposta.content)

        dados_diarios.append({
            "Número": numero,
            "Data": data,
            "Título do Arquivo": titulo_arquivo,
            "URL do Arquivo": url_arquivo,
            "Caminho do Arquivo": caminho_arquivo
        })

        print(numero, data, url_arquivo, titulo_arquivo)

    if soup.find(class_="fa-chevron-right"):
        url_proxima_pagina = URL_BASE + soup.find(class_="fa-chevron-right").parent["href"]

        print(url_proxima_pagina)
        resposta = sessao.get(url_proxima_pagina)
    else:
        break

    # Remover/comentar o if e o break para coletar todos os dados
    if contador > 1:
        break

    contador += 1

In [ ]:
caminho_relatorio = os.path.join(CAMINHO_RAIZ, f"coleta_dados_diarios_{DATA_LOG}.csv")

pd.DataFrame(dados_diarios).to_csv(caminho_relatorio, index=False)

In [ ]:
pd.DataFrame(dados_diarios)